# Fine-tune SmolVLA on ALOHA sim — Colab training notebook

**Current mission: SmolVLA vs ACT on transfer cube.** Fine-tune `lerobot/smolvla_base` on
`aloha_sim_transfer_cube_human` (cell 5 already selects it) and compare against the measured
ACT baseline: **13/20 = 65 % success** (20-episode protocol, matched mujoco 2.3.7).
Latency table is pre-filled locally: ACT 0.68 ms / 1474 Hz / 266 MB vs SmolVLA 27.7 ms / 36 Hz / 927 MB.

**A100 session — run in this order** (fresh VM every time you switch runtime type):

| # | cell | what | notes |
|---|---|---|---|
| 1 | GPU check | expect `NVIDIA A100...` | |
| 2 | Drive mount | tarballs + checkpoints live on Drive | |
| 3 | install | lerobot 0.4.4 (+ torch swap) ~3 min | |
| 4 | 3b | HF token (getpass) | rate-limit lift for metadata |
| 5 | 4b | wandb key (getpass) | live loss charts |
| 6 | **5** | hyperparameters | prints `GPU=... -> BATCH_SIZE=64` |
| 7 | **3c** | stage dataset + models from Drive tarballs | finds them in ANY Drive folder |
| 8 | **5b** | FULL RUN ~4–5 h | skip 5a — pipeline already validated on T4 |
| 9 | 7 | zip checkpoint -> Drive | then eval locally (last cell) |

Training is pure behavior cloning from the dataset — no MuJoCo needed here. Checkpoints save
every `SAVE_FREQ` steps to Drive; a disconnect costs at most that many steps (resume cell 6).


In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Google Drive — tarballs are read from here; checkpoints persist here across disconnects
from google.colab import drive
drive.mount('/content/drive')
# (OUTPUT_DIR is set in cell 5, derived from the chosen dataset)


In [ ]:
# 3. Install — SAME lerobot version as the project's Docker image (0.4.4), so the
#    checkpoint drops straight into `docker compose run --rm eval` back home.
#    Training uses the pyav video backend, so a torch/torchcodec mismatch is harmless.
%pip install -q "lerobot[smolvla]==0.4.4" av
import lerobot, torch
print("lerobot", lerobot.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# 3b. HF Hub download reliability.
# The dataset lives in HF Xet storage; the HTTP fallback bridge can 403 ("invalid key pair
# id"), so install the NATIVE xet client. IMPORTANT: keep huggingface_hub inside lerobot's
# pin (<0.36) — a blanket upgrade to hub 1.x breaks lerobot and transformers.
%pip install -q "huggingface_hub[cli,hf-transfer]>=0.34.2,<0.36" hf_xet
import os, torch
os.environ.pop("HF_HUB_DISABLE_XET", None)    # let the native xet client handle downloads
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())  # sanity after installs
from getpass import getpass
_tok = getpass("HF token (hf.co/settings/tokens, Enter to skip): ").strip()
if _tok: os.environ["HF_TOKEN"] = _tok; print("HF_TOKEN set")


In [ ]:
# 5. HYPERPARAMETERS — the single place to edit. Dry run and full run read from here.

# ---- dataset selection (tarball of the same name must be somewhere in your Drive) ----
DATASET = "lerobot/aloha_sim_transfer_cube_human"  # cube handover -> head-to-head vs the 80% ACT baseline
#         "lerobot/aloha_sim_insertion_human"        # peg insertion (bimanual, 14-D)
#         "lerobot/svla_so101_pickplace"             # SO-101 pick-place (single arm, 6-D)
RENAME_MAPS = {   # dataset camera names -> smolvla_base's camera slots
    "lerobot/aloha_sim_insertion_human":     '{"observation.images.top": "observation.images.camera1"}',
    "lerobot/aloha_sim_transfer_cube_human": '{"observation.images.top": "observation.images.camera1"}',
    "lerobot/svla_so101_pickplace":          '{"observation.images.up": "observation.images.camera1", "observation.images.side": "observation.images.camera2"}',
}
RENAME = RENAME_MAPS[DATASET]
SHORT = DATASET.split("/")[-1]
OUTPUT_DIR = f"/content/drive/MyDrive/train_{SHORT}"   # checkpoints land here (Drive)

# ---- dry run (cheap GPU, e.g. T4 16 GB): validates the pipeline, quality irrelevant ----
DRY_BATCH_SIZE = 32      # fits a T4; SmolVLA needs ~11.5 GB at bs 44, so 64 OOMs on 16 GB
DRY_STEPS      = 200
DRY_SAVE_FREQ  = 100

# ---- full run: batch auto-sized to the GPU (A100 -> 64 paper recipe; T4 -> 32 fits 16 GB) ----
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64 if ("A100" in GPU or "H100" in GPU or "L40" in GPU) else 32
STEPS      = 20000
SAVE_FREQ  = 2000
print(f"GPU={GPU} -> BATCH_SIZE={BATCH_SIZE}")   # A100 @64: ~4-5 h | T4 @32: ~30 h (use resume)

# NOTE: switching runtime type (T4 <-> A100) gives a FRESH VM: re-run cells 1-4b and 3c.
print(f"dataset={DATASET}  output={OUTPUT_DIR}")


In [ ]:
# 3c. Stage the SELECTED dataset + models from Drive tarballs (zero HF network).
# Upload to drive.google.com (any folder): smolvla_models_cache.tar.gz (~1.6 GB, once)
# and the tarball of the dataset you picked in cell 5 (data/<name>.tar.gz from the repo).
# RUN CELL 5 (hyperparameters) FIRST so DATASET/SHORT are defined.
import glob, os

def find(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    assert hits, f"{name} not found in MyDrive — upload it to drive.google.com first"
    return hits[0]

DATA_TAR   = find(f"{SHORT}.tar.gz")
MODELS_TAR = find("smolvla_models_cache.tar.gz")
print("dataset tar:", DATA_TAR); print("models tar :", MODELS_TAR)

!mkdir -p /root/.cache/huggingface/lerobot/lerobot /root/.cache/huggingface/hub
!rm -rf /root/.cache/huggingface/lerobot/lerobot/{SHORT}
!tar -xzf "{DATA_TAR}"   -C /root/.cache/huggingface/lerobot/lerobot/
!tar -xzf "{MODELS_TAR}" -C /root/.cache/huggingface/hub/
!du -sh /root/.cache/huggingface/lerobot/lerobot/{SHORT} /root/.cache/huggingface/hub/models--*
print("ALL STAGED — run 5a (dry) or 5b (full)")


In [ ]:
# 4b. Live loss curves via Weights & Biases (free account: wandb.ai)
# Key: Colab Secrets (browser UI) -> getpass prompt (VS Code bridge & UI). Never hardcode it.
USE_WANDB = True
WANDB_FLAGS = ""
if USE_WANDB:
    try:
        from google.colab import userdata
        KEY = userdata.get("WANDB_API_KEY")     # browser Colab UI only
    except Exception:
        from getpass import getpass
        KEY = getpass("wandb API key (from wandb.ai/authorize): ").strip()
    import wandb
    assert wandb.login(key=KEY, relogin=True), "wandb login failed — check the key"
    WANDB_FLAGS = "--wandb.enable=true --wandb.project=smolvla-edge"
    print("wandb: logged in OK")


## 5a. Dry run first (~3–5 min) — validate EVERYTHING before spending 4 GPU-hours

200 steps with the identical config. What to check off while it runs:

- [ ] dataset downloads + first batch loads (no rename/backend errors)
- [ ] a **wandb run URL** is printed → open it → `train/loss` chart is updating live
- [ ] console shows `loss:` decreasing (≈0.3–0.4 by step 200, mirroring the local smoke run)
- [ ] a checkpoint appears under `$OUTPUT_DIR-dryrun/checkpoints/000100/`
- [ ] `train_console.log` exists on Drive

All boxes ticked → run the full cell (5b). The dry run writes to a separate `-dryrun`
dir, so the real run starts clean.

In [ ]:
# 5. HYPERPARAMETERS — the single place to edit. Dry run and full run read from here.

# ---- dataset selection (tarball of the same name must be somewhere in your Drive) ----
DATASET = "lerobot/aloha_sim_transfer_cube_human"  # cube handover -> head-to-head vs the 80% ACT baseline
#         "lerobot/aloha_sim_insertion_human"        # peg insertion (bimanual, 14-D)
#         "lerobot/svla_so101_pickplace"             # SO-101 pick-place (single arm, 6-D)
RENAME_MAPS = {   # dataset camera names -> smolvla_base's camera slots
    "lerobot/aloha_sim_insertion_human":     '{"observation.images.top": "observation.images.camera1"}',
    "lerobot/aloha_sim_transfer_cube_human": '{"observation.images.top": "observation.images.camera1"}',
    "lerobot/svla_so101_pickplace":          '{"observation.images.up": "observation.images.camera1", "observation.images.side": "observation.images.camera2"}',
}
RENAME = RENAME_MAPS[DATASET]
SHORT = DATASET.split("/")[-1]
OUTPUT_DIR = f"/content/drive/MyDrive/train_{SHORT}"   # checkpoints land here (Drive)

# ---- dry run (cheap GPU, e.g. T4 16 GB): validates the pipeline, quality irrelevant ----
DRY_BATCH_SIZE = 32      # fits a T4; SmolVLA needs ~11.5 GB at bs 44, so 64 OOMs on 16 GB
DRY_STEPS      = 200
DRY_SAVE_FREQ  = 100

# ---- full run: batch auto-sized to the GPU (A100 -> 64 paper recipe; T4 -> 32 fits 16 GB) ----
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64 if ("A100" in GPU or "H100" in GPU or "L40" in GPU) else 32
STEPS      = 20000
SAVE_FREQ  = 2000
print(f"GPU={GPU} -> BATCH_SIZE={BATCH_SIZE}")   # A100 @64: ~4-5 h | T4 @32: ~30 h (use resume)

# NOTE: switching runtime type (T4 <-> A100) gives a FRESH VM: re-run cells 1-4b and 3c.
print(f"dataset={DATASET}  output={OUTPUT_DIR}")


In [ ]:
# 5b. FULL RUN (~4-5 h on A100 at batch 64). Uses the DATASET picked in cell 5.
# A leftover dir from an aborted start (no checkpoints yet) blocks lerobot-train.
# Wipe it ONLY if it holds no checkpoints; with checkpoints, use the resume cell instead.
import os, glob as _g
if os.path.isdir(OUTPUT_DIR):
    if _g.glob(f"{OUTPUT_DIR}/checkpoints/*"):
        raise SystemExit(f"{OUTPUT_DIR} has checkpoints — use the RESUME cell, not a fresh 5b run")
    print("clearing checkpoint-less leftover dir")
    !rm -rf {OUTPUT_DIR}
!mkdir -p {OUTPUT_DIR}
cmd = (
    "lerobot-train"
    " --policy.path=lerobot/smolvla_base"
    " --policy.push_to_hub=false"
    f" --dataset.repo_id={DATASET}"
    " --dataset.video_backend=pyav"
    f" --rename_map='{RENAME}'"
    f" --batch_size={BATCH_SIZE} --steps={STEPS} --save_freq={SAVE_FREQ} {WANDB_FLAGS}"
    f" --output_dir={OUTPUT_DIR} 2>&1 | tee -a {OUTPUT_DIR}/train_console.log"
)
print(cmd)
!{cmd}
# Loss history is NOT saved by lerobot; the tee keeps it on Drive. Extract later with:
#   grep -oE "step:[0-9]+ .*loss:[0-9.]+" train_console.log


In [ ]:
# 6. If the runtime disconnected mid-run: reconnect, re-run cells 1-5 and 3c, then resume
#    from the last saved checkpoint instead of re-running 5b:
# cmd = (f"lerobot-train --config_path={OUTPUT_DIR}/checkpoints/last/pretrained_model/train_config.json"
#        f" --resume=true 2>&1 | tee -a {OUTPUT_DIR}/train_console.log")
# print(cmd)
# !{cmd}


In [ ]:
# 7. Package the final checkpoint for the trip home (skip if you pull from Drive directly).
!ls -la {OUTPUT_DIR}/checkpoints/
ZIP = f"/content/smolvla_{SHORT}_last.zip"
!cd {OUTPUT_DIR}/checkpoints && zip -qr {ZIP} last/ && ls -la {ZIP}
# then: download via the Files panel, or copy into Drive:
!cp {ZIP} /content/drive/MyDrive/ && echo "zip copied to Drive"


## Back on the dev box — the head-to-head

```bash
# unzip your checkpoint into the repo (transfer-cube run):
unzip smolvla_aloha_sim_transfer_cube_human_last.zip -d outputs/train/smolvla_transfer_cube/checkpoints/

# YOUR SmolVLA, official protocol (20 episodes, dataset's own instruction):
docker compose run --rm shell python -m smolvla_edge.eval --mode sim \
    --policy-path outputs/train/smolvla_transfer_cube/checkpoints/last \
    --env-id gym_aloha/AlohaTransferCube-v0 --episodes 20 \
    --task "Pick up the cube with the right arm and transfer it to the left arm."

# ACT baseline, same protocol (already measured locally — see tasks.md):
docker compose run --rm shell python -m smolvla_edge.eval --mode sim \
    --policy-path lerobot/act_aloha_sim_transfer_cube_human \
    --env-id gym_aloha/AlohaTransferCube-v0 --episodes 20 --task ""

# demo GIF with YOUR policy:
docker compose run --rm shell python scripts/make_demo_gif.py --mode rollout \
    --policy-path outputs/train/smolvla_transfer_cube/checkpoints/last \
    --env-id gym_aloha/AlohaTransferCube-v0 \
    --task "Pick up the cube with the right arm and transfer it to the left arm."
```

Checkpoint facts (identical regardless of training GPU): 450 M params, ~865 MB, ships its
processor pipeline (rename/tokenize/normalize) — the local eval harness uses it automatically.
